# 01 – Data: van Roboflow-export naar train / val / test

Dit notebook neemt de gelabelde frames uit Roboflow en verdeelt ze over **train**, **val** en **test**.
Het resultaat is een map `dataset/` (naast dit notebook) die `02_model.ipynb` gebruikt om te trainen.

Inhoud:
1. Wat we in Roboflow gedaan hebben (en waarom)
2. Instellingen en paden
3. De gelabelde frames inlezen
4. Hoe groot is de bal? (bepaalt de image size van het model)
5. Dubbele frames in de video
6. De split: waarom op tijd en niet willekeurig
7. De dataset aanmaken
8. Controle van het resultaat
9. Conclusie en beperkingen

> Kernel: de gewone `venv` volstaat (geen GPU nodig). `data/` wordt nooit aangepast: alles wat dit notebook maakt, komt in `kobe/notebooks/dataset/`.

## 1. Wat we in Roboflow gedaan hebben

**Bron.** Een broadcastclip van PSG – Inter (Champions League-finale, München): 22 seconden, 30 fps, 1920×1080 = **660 frames**.
Alle frames staan in `data/frames/` (gemaakt met `scripts/extract_frames.py`).

**Roboflow-project:** `git-push-and-pray/object-detection-sjoav`, **versie 2** (30/10/2026), **100 gelabelde frames**.

**Klassen (5):**

| id | klasse | wat | waarom een eigen klasse |
|---|---|---|---|
| 0 | `ball` | de wedstrijdbal | het eigenlijke doel: een klein, snel object dat een voorgetraind (COCO) model bijna nooit vindt |
| 1 | `player_A` | veldspeler PSG (donkerblauw) | het model geeft meteen het team, nodig voor de minimap |
| 2 | `player_B` | veldspeler Inter (geel) | idem |
| 3 | `player_B_goal_keeper` | keeper Inter (oranje) | ander shirt dan zijn team; zou anders als fout geteld worden |
| 4 | `referee` | scheidsrechter en lijnrechters | hoort bij geen enkel team |

De keeper van PSG staat in deze clip niet in beeld, daarom is er geen `player_A_goal_keeper`.

**Hoe we gelabeld hebben:**
- **Handmatig** boxen tekenen, en
- **Label Assist** van Roboflow: een model stelt boxen voor, wij controleren en corrigeren ze.
  Dat gaat veel sneller voor de ±20 spelers per frame. Het risico is dat je fouten van het hulpmodel overneemt
  (een box te ruim, een verkeerd team, een gemiste bal). Daarom is elke voorgestelde box nagekeken, en de bal,
  die Label Assist meestal mist, is altijd met de hand gelabeld.

**Labelregels** (uit `docs/labeling_protocol.md`, zodat iedereen in het team hetzelfde doet):
- de box omsluit **strak** het object; **schaduwen** worden nooit meegelabeld;
- **motion blur**: de box omvat de volledige uitgerekte veeg van de bal;
- **occlusie**: ≥ 30 % zichtbaar → labelen (box rond het zichtbare deel); < 30 % → niet labelen;
- **rand van het beeld**: alleen labelen als meer dan de helft zichtbaar is.

**Export-instellingen** (zie `data/dataset_yolo/README.roboflow.txt`):
- formaat YOLOv8 (één `.txt` per beeld: `klasse x_midden y_midden breedte hoogte`, genormaliseerd 0–1);
- **geen preprocessing**: geen resize, de beelden blijven 1920×1080. Verkleinen zou de bal (±17 px) nog kleiner maken;
- **geen augmentatie** in Roboflow. YOLO augmenteert zelf tijdens het trainen (elke epoch andere varianten).
  Augmentatie in Roboflow maakt bovendien kopieën *vóór* de split: dan kan een kopie van een train-beeld in val belanden (leakage);
- alle beelden staan in `train/` van de export: **Roboflow's eigen split gebruiken we niet**, we splitsen zelf op tijd (sectie 6).

**Peer review:** na het labelen hebben de teamleden elkaars frames steekproefsgewijs nagekeken, informeel en zonder vast protocol. We hebben daar bewust beperkt tijd in gestoken: spelers, keeper en scheidsrechters zijn groot en duidelijk zichtbaar, en Label Assist zorgde al voor een consistente basis. De grootste kans op labelfouten zit bij de **bal** (klein, motion blur, gedeeltelijk verborgen). Een systematische controle van de bal-labels is daarom het belangrijkste verbeterpunt voor een volgende labelronde.

In [ ]:
import re
import shutil
from collections import Counter
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import yaml

## 2. Instellingen en paden

Het notebook zoekt zelf de hoofdmap van het project (de map met `data/dataset_yolo`), zodat het werkt
ongeacht waar je de repo hebt staan.

In [ ]:
NB_DIR = Path.cwd()                                   # kobe/notebooks/
ROOT = next(p for p in [NB_DIR, *NB_DIR.parents] if (p / "data" / "dataset_yolo").exists())
EXPORT_DIR = ROOT / "data" / "dataset_yolo"           # Roboflow export (read-only)
FRAMES_DIR = ROOT / "data" / "frames"                 # all 660 frames of the clip (read-only)
DATASET_DIR = NB_DIR / "dataset"                      # output of this notebook

N_FRAMES, FPS = 660, 30
print("project root:", ROOT)
print("output      :", DATASET_DIR)

## 3. De gelabelde frames inlezen

Elke bestandsnaam bevat de oorspronkelijke frame-index, bv. `frame_0042_idx_00042_jpg.rf.<hash>.jpg` → frame 42.
Die index hebben we nodig om op **tijd** te kunnen splitsen.

In [ ]:
names = yaml.safe_load((EXPORT_DIR / "data.yaml").read_text())["names"]


def frame_idx(path):
    """Original frame index from a name like frame_0042_idx_00042_jpg.rf.<hash>.jpg"""
    return int(re.search(r"idx_(\d+)", Path(path).name).group(1))


# labeled[idx] = (image path, label path, Counter {class id: number of boxes})
labeled = {}
for img in sorted(EXPORT_DIR.glob("*/images/*.jpg")):
    label = img.parent.parent / "labels" / (img.stem + ".txt")
    ids = [int(line.split()[0]) for line in label.read_text().splitlines() if line.strip()]
    labeled[frame_idx(img)] = (img, label, Counter(ids))

total = sum((c for _, _, c in labeled.values()), Counter())
print(f"{len(labeled)} labeled frames, {sum(total.values())} boxes\n")
print(f"{'class':<22}{'boxes':>7}{'per frame':>11}{'frames with it':>16}")
for cid, name in enumerate(names):
    frames_with = sum(c[cid] > 0 for _, _, c in labeled.values())
    print(f"{name:<22}{total[cid]:>7}{total[cid] / len(labeled):>11.2f}{frames_with:>16}")

**Wat zien we?** Spelers zijn er genoeg (±10 per team per frame). De zeldzame klassen zijn de **bal**
(niet in elk frame zichtbaar: verstopt achter een speler of < 30 % zichtbaar → volgens de regels niet gelabeld)
en de **keeper** (één per frame). Die bepalen hoe goed het model kan leren en hoe betrouwbaar de evaluatie is.

## 4. Hoe groot is de bal?

YOLO verkleint elk beeld naar `imgsz` × `imgsz` pixels voordat het naar het netwerk gaat. De fijnste feature map
van YOLO heeft een stride van 8: één cel ziet 8×8 pixels van het (verkleinde) beeld. Een object van maar enkele
pixels valt dan grotendeels binnen één cel en is moeilijk te vinden.

In [ ]:
ball_wh = []
for img, label, _ in labeled.values():
    for line in label.read_text().splitlines():
        parts = line.split()
        if parts and int(parts[0]) == names.index("ball"):
            ball_wh.append((float(parts[3]) * 1920, float(parts[4]) * 1080))  # normalised -> pixels
ball_wh = np.array(ball_wh)
size = ball_wh.max(axis=1)  # longest side of the box
print(f"{len(size)} balls, longest side in the 1920x1080 frame: "
      f"median {np.median(size):.1f} px, min {size.min():.1f} px, max {size.max():.1f} px\n")
print(f"{'imgsz':>6}{'scale':>8}{'median ball':>13}{'smallest ball':>15}")
for imgsz in (640, 1280, 1600, 1920):
    scale = imgsz / 1920
    print(f"{imgsz:>6}{scale:>8.2f}{np.median(size) * scale:>11.1f}px{size.min() * scale:>13.1f}px")

plt.figure(figsize=(6, 2.5))
plt.hist(size, bins=15)
plt.xlabel("longest side of the ball box (px, original frame)")
plt.ylabel("balls")
plt.title("Ball size in the labeled frames")
plt.tight_layout()
plt.show()

**Conclusie:** bij `imgsz 640` (de standaard van YOLO) wordt de bal ±6 px groot: in onze eerste smoke test werd hij
daardoor nooit gevonden. Bij **`imgsz 1920`** (de originele resolutie) blijft de bal ±17 px (de kleinste 9 px) en gaat er geen
informatie verloren. Dat is de keuze voor `02_model.ipynb`, ook al kost het meer rekentijd en geheugen.

## 5. Dubbele frames in de video

De clip is omgezet van 25 naar 30 fps: daarbij wordt ongeveer elk 6e frame een kopie van het vorige.
Een frame dat (bijna) identiek is aan het vorige noemen we een duplicaat. Twee keer hetzelfde beeld in de
dataset voegt niets toe, en als ze in verschillende splits zouden belanden, is dat leakage.

In [ ]:
def duplicate_frames(threshold=0.5):
    """Frame indices that are (almost) identical to the previous frame (mean grey difference < threshold)."""
    duplicates, prev = set(), None
    for path in sorted(FRAMES_DIR.glob("*.jpg"), key=frame_idx):
        small = cv2.cvtColor(cv2.resize(cv2.imread(str(path)), (480, 270)), cv2.COLOR_BGR2GRAY).astype(np.float32)
        if prev is not None and np.abs(small - prev).mean() < threshold:
            duplicates.add(frame_idx(path))
        prev = small
    return duplicates


duplicates = duplicate_frames()
labeled_dups = sorted(i for i in labeled if i in duplicates and i - 1 in labeled)
print(f"{len(duplicates)} of {N_FRAMES} frames are a copy of the previous frame")
print(f"labeled frames that are a copy of another labeled frame (dropped): {labeled_dups}")

## 6. De split: waarom op tijd en niet willekeurig

**Probleem met een willekeurige split:** opeenvolgende frames zijn bijna identiek (1/30 s ertussen). Als frame 41 in
train zit en frame 42 in val, heeft het model het val-beeld eigenlijk al gezien. De val-score is dan te mooi
en zegt niets over nieuwe beelden. Dat heet **leakage**.

**Oplossing:** we splitsen op **tijdsblokken**. Elk blok van de clip hoort volledig bij één split, en tussen een
train-blok en een val/test-blok laten we een **gap** van 10 frames (1/3 s) die nergens gebruikt wordt.

**Drie splits:**
- **train**: hierop leert het model;
- **val**: tijdens het trainen gebruikt om te kiezen wanneer we stoppen (early stopping) en welk epoch het beste is (`best.pt`);
- **test**: pas helemaal op het einde, één keer. Omdat val al gebruikt is om keuzes te maken, is enkel de test-score
  een eerlijke schatting voor nieuwe beelden.

**Afwijking van de README in `kobe/training/`.** Daar stonden blokken van 60 frames, met val en test op vaste plaatsen.
Maar het team heeft vooral rond **frame 0–62, 200–245 en 525–658** gelabeld. Met die vaste blokken zou val maar
**4 frames** krijgen: onbruikbaar. Daarom leggen we de blokken nu op de gelabelde momenten:

| blok | frames | split | waarom |
|---|---|---|---|
| 1 | 0–149 | train | begin van de clip (camera rond de middencirkel) |
| 2 | 150–299 | **val** | een volledig apart moment in het midden, met de meeste gelabelde ballen |
| 3 | 300–599 | train | einde van de aanval (camera bij het strafschopgebied): train ziet zo beide camerastanden |
| 4 | 600–659 | **test** | het laatste stuk, nooit gezien tijdens het trainen of kiezen |

Elke frame van de clip hoort zo bij een split: als het team later extra frames labelt, vallen die vanzelf op de juiste plek.

In [ ]:
BLOCKS = [(0, 149, "train"), (150, 299, "val"), (300, 599, "train"), (600, 659, "test")]
GAP = 10  # train frames this close to a val/test block are not used


def block_split(idx):
    return next(split for start, end, split in BLOCKS if start <= idx <= end)


def split_of(idx):
    """'train', 'val', 'test' or 'gap'. Only train frames are dropped, so val and test stay complete."""
    split = block_split(idx)
    if split == "train":
        near = range(max(0, idx - GAP), min(N_FRAMES, idx + GAP + 1))
        if any(block_split(j) != "train" for j in near):
            return "gap"
    return split


# Timeline: which part of the clip goes where, and where the labels are
colours = {"train": "tab:blue", "val": "tab:orange", "test": "tab:green", "gap": "grey"}
fig, ax = plt.subplots(figsize=(12, 1.8))
for idx in range(N_FRAMES):
    ax.axvspan(idx, idx + 1, color=colours[split_of(idx)], alpha=0.25, lw=0)
ax.vlines(sorted(labeled), 0, 1, color="black", lw=1)
for split, colour in colours.items():
    ax.plot([], [], color=colour, lw=6, alpha=0.5, label=split)
ax.plot([], [], color="black", lw=1, label="labeled frame")
ax.set(xlim=(0, N_FRAMES), yticks=[], xlabel="frame index", title="Split of the clip")
ax.legend(loc="upper center", ncol=5, fontsize=8, bbox_to_anchor=(0.5, -0.45))
plt.tight_layout()
plt.show()

## 7. De dataset aanmaken

Voor elk gelabeld frame: bepaal de split, sla gap-frames en duplicaten over, en kopieer beeld + label naar
`dataset/<split>/images` en `dataset/<split>/labels`. Daarnaast schrijven we `dataset/data.yaml`: het bestand dat
ultralytics leest om te weten waar de beelden staan en welke klassen er zijn.

De map `dataset/` wordt elke keer opnieuw opgebouwd uit de export, zodat ze altijd overeenkomt met dit notebook.

In [ ]:
SPLITS = ["train", "val", "test"]
if DATASET_DIR.exists():
    shutil.rmtree(DATASET_DIR)
for split in SPLITS:
    for sub in ["images", "labels"]:
        (DATASET_DIR / split / sub).mkdir(parents=True)

dropped = {"gap": [], "duplicate": []}
used = {split: [] for split in SPLITS}
for idx, (img, label, _) in sorted(labeled.items()):
    split = split_of(idx)
    if split == "gap":
        dropped["gap"].append(idx)
        continue
    if idx in labeled_dups:
        dropped["duplicate"].append(idx)
        continue
    shutil.copy2(img, DATASET_DIR / split / "images" / img.name)
    shutil.copy2(label, DATASET_DIR / split / "labels" / label.name)
    used[split].append(idx)

config = {"path": str(DATASET_DIR), "train": "train/images", "val": "val/images", "test": "test/images",
          "nc": len(names), "names": names}
(DATASET_DIR / "data.yaml").write_text(yaml.safe_dump(config, sort_keys=False))
print("written:", DATASET_DIR / "data.yaml")
print("dropped (gap)      :", dropped["gap"])
print("dropped (duplicate):", dropped["duplicate"])

## 8. Controle van het resultaat

In [ ]:
counts = {s: sum((labeled[i][2] for i in used[s]), Counter()) for s in SPLITS}
print(f"{'':<22}" + "".join(f"{s:>8}" for s in SPLITS))
print(f"{'frames':<22}" + "".join(f"{len(used[s]):>8}" for s in SPLITS))
for cid, name in enumerate(names):
    print(f"{name:<22}" + "".join(f"{counts[s][cid]:>8}" for s in SPLITS))
n = sum(len(v) for v in used.values())
print(f"\nshare of frames: " + ", ".join(f"{s} {100 * len(used[s]) / n:.0f}%" for s in SPLITS))
for s in SPLITS:
    for cid, name in enumerate(names):
        if counts[s][cid] < 10:
            print(f"WARNING: only {counts[s][cid]} '{name}' in {s}: metrics for this class are not reliable")

## 9. Conclusie en beperkingen

- **De split is op tijd**, met een gap tussen train en val/test, en zonder dubbele frames: geen leakage.
- **Train ziet beide camerastanden** (middencirkel en strafschopgebied); val en test zijn elk een apart moment.
- **De test-set bevat weinig ballen** (zie de waarschuwingen hierboven). Dat is geen labelfout: frame 600–659
  ligt **na het doelpunt**, de bal ligt dan in het doel en is niet meer zichtbaar. Een bal die het model daar
  toch vindt, is dus altijd een false positive. Elke gemiste of extra bal verandert de test-score voor de
  bal sterk; voor de spelers is de test-set wel groot genoeg. Verbetering: extra frames labelen in frame 600–659, en ook in 300–449,
  waar nog niets gelabeld is (die komen dan in train).
- **Label Assist** versnelt het labelen, maar kan fouten van het hulpmodel binnensmokkelen. Een peer review
  (sectie 1) is de manier om dat te controleren.

Volgende stap: `02_model.ipynb` traint het model op `dataset/data.yaml`.